In [0]:
# ============================================================
# NOTEBOOK : DIM_SUPPLIER
# PURPOSE  : SUPPLIER DIMENSION INCREMENTAL LOAD
# ============================================================

from pyspark.sql.functions import *
from delta.tables import *
import uuid


In [0]:
%run /Users/ragul.p.dev@gmail.com/fabric_incremental_project/Functions/FN_Common_Functions

In [0]:
%run "/Users/ragul.p.dev@gmail.com/fabric_incremental_project/Functions/FN_LOGGER"

Max Date updated successfully


FN_LOGGER LOADED SUCCESSFULLY


In [0]:
# ============================================================
# GET BRONZE DATA
# ============================================================

try:

    metadata = get_metadata("supplier_tbl")

    table_name = metadata["target_table"]

    source_system = metadata["source_system"]

    watermark_column = metadata["watermark_column"]

    primary_key = metadata["primary_key_column"]

    source_table = f"bronze.{table_name}"

    target_table = "silver.dim_supplier"

    pipeline_name = "PL_DIM_SUPPLIER"

    pipeline_run_id = str(uuid.uuid4())

    start_time = get_current_timestamp()

    last_watermark = get_watermark(table_name)

    print(f"Last Watermark : {last_watermark}")

    bronze_df = spark.sql(f"""

        SELECT

            supplier_id,
            supplier_name,
            contact_name,
            email,
            phone,
            city,
            country,
            created_date,
            modified_date

        FROM {source_table}

        WHERE {watermark_column} > '{last_watermark}'

    """)

    bronze_df.createOrReplaceTempView(
        "vw_bronze_supplier"
    )

    print(f"Bronze View Created : {table_name}")

    rows_read = bronze_df.count()

    print(f"Rows Read : {rows_read}")

except Exception as e:

    print(f"Bronze View Creation Failed : {table_name}")

    raise(e)



Last Watermark : 1900-01-01 00:00:00
Bronze View Created : supplier_tbl
Rows Read : 5


FN_COMMON_FUNCTIONS LOADED SUCCESSFULLY


In [0]:
# ============================================================
# CREATE SILVER VIEW
# ============================================================

try:

    silver_df = spark.sql("""

        SELECT

            supplier_id,

            upper(trim(supplier_name))
                AS supplier_name,

            initcap(contact_name)
                AS contact_name,

            lower(email)
                AS email,

            phone,

            initcap(city)
                AS city,

            upper(country)
                AS country,

            CASE

                WHEN upper(country) = 'INDIA'
                THEN 'DOMESTIC'

                ELSE 'INTERNATIONAL'

            END AS supplier_category,

            modified_date,

            sha2(
                concat_ws(
                    '|',
                    supplier_name,
                    contact_name,
                    email,
                    city,
                    country
                ),
                256
            ) AS hash_key,

            current_timestamp()
                AS effective_start_date,

            CAST(NULL AS TIMESTAMP)
                AS effective_end_date,

            1 AS is_current,

            0 AS is_deleted

        FROM vw_bronze_supplier

    """)

    silver_df.createOrReplaceTempView(
        "vw_silver_supplier"
    )

    print(f"Silver View Created : {table_name}")

except Exception as e:

    print(f"Silver View Creation Failed : {table_name}")

    raise(e)



Silver View Created : supplier_tbl


In [0]:
# ============================================================
# CREATE TARGET TABLE
# ============================================================

try:

    spark.sql("""

        CREATE TABLE IF NOT EXISTS silver.dim_supplier
        (
            supplier_id BIGINT,
            supplier_name STRING,
            contact_name STRING,
            email STRING,
            phone STRING,
            city STRING,
            country STRING,
            supplier_category STRING,
            modified_date TIMESTAMP,
            hash_key STRING,
            effective_start_date TIMESTAMP,
            effective_end_date TIMESTAMP,
            is_current INT,
            is_deleted INT
        )

        USING DELTA

    """)

    print(f"Target Table Created : {target_table}")

except Exception as e:

    print(f"Target Table Creation Failed : {target_table}")

    raise(e)



Target Table Created : silver.dim_supplier


In [0]:
# ============================================================
# MERGE LOGIC
# ============================================================

try:

    spark.sql("""

        MERGE INTO silver.dim_supplier AS target

        USING vw_silver_supplier AS source

        ON target.supplier_id = source.supplier_id
           AND target.is_current = 1

        WHEN MATCHED
             AND target.hash_key <> source.hash_key

        THEN UPDATE SET

            target.effective_end_date =
                current_timestamp(),

            target.is_current = 0

        WHEN NOT MATCHED

        THEN INSERT
        (
            supplier_id,
            supplier_name,
            contact_name,
            email,
            phone,
            city,
            country,
            supplier_category,
            modified_date,
            hash_key,
            effective_start_date,
            effective_end_date,
            is_current,
            is_deleted
        )

        VALUES
        (
            source.supplier_id,
            source.supplier_name,
            source.contact_name,
            source.email,
            source.phone,
            source.city,
            source.country,
            source.supplier_category,
            source.modified_date,
            source.hash_key,
            source.effective_start_date,
            source.effective_end_date,
            source.is_current,
            source.is_deleted
        )

    """)

    print(f"Merge Completed : {table_name}")

    spark.sql("""

        UPDATE silver.dim_supplier

        SET

            is_deleted = 1,
            is_current = 0,
            effective_end_date = current_timestamp()

        WHERE supplier_id NOT IN
        (
            SELECT supplier_id
            FROM vw_silver_supplier
        )

        AND is_current = 1

    """)

    print(f"Soft Delete Completed : {table_name}")

except Exception as e:

    print(f"Merge Failed : {table_name}")

    raise(e)



Merge Completed : supplier_tbl
Soft Delete Completed : supplier_tbl


In [0]:
# ============================================================
# UPDATE WATERMARK & AUDIT LOG
# ============================================================

try:

    max_date = get_max_date(
        bronze_df,
        watermark_column
    )

    if max_date is not None:

        update_watermark(
            table_name,
            max_date
        )

        print(f"Watermark Updated : {table_name}")

    else:

        print("No Incremental Records Found")

    rows_written = silver_df.count()

    end_time = get_current_timestamp()

    execution_time_seconds = int(
        (end_time - start_time).total_seconds()
    )

    insert_audit_log(

        pipeline_run_id,
        pipeline_name,
        source_system,
        table_name,
        start_time,
        end_time,
        rows_read,
        rows_written,
        "SUCCESS",
        execution_time_seconds

    )

    print(f"DIM_SUPPLIER SUCCESSFULLY LOADED : {table_name}")

except Exception as e:

    insert_error_log(

        str(uuid.uuid4()),
        pipeline_run_id,
        table_name,
        source_system,
        "DIM_SUPPLIER",
        str(e)

    )

    print(f"DIM_SUPPLIER LOAD FAILED : {table_name}")

    raise(e)

Watermark Updated : supplier_tbl
Watermark Updated : supplier_tbl
Audit Log Inserted : supplier_tbl
DIM_SUPPLIER SUCCESSFULLY LOADED : supplier_tbl
